In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.signal import savgol_filter, butter, filtfilt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
%load_ext rpy2.ipython

Import data, set plotting params

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    "svg.fonttype": "none",   
    "pdf.fonttype": 42,       
    "ps.fonttype": 42,      
    "text.usetex": False,    
})

In [ ]:
def lowpass(x, fs, cutoff_hz=0.3, order=3):
    b, a = butter(order, cutoff_hz / (fs / 2.0), btype="low")
    return filtfilt(b, a, np.asarray(x, float))

In [ ]:
df = pd.read_csv(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\ResidualTrialDF_Reward.csv')
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = df[df['Animal'].isin(goodanimals)]
df['Site'] = df['Region']
df['Region'] = df['Site'].str[:2]

df = (df.sort_values(["Trial","Animal","Site","Time to Shelter"],
                     kind="mergesort", na_position="last")
        .reset_index(drop=True))

df['Outcome'] = np.where(df['Time to Reward'].notna(), 'Rewarded', 'Unrewarded')
df['MonsterCharge'] = np.where(df['Time to Monster'].notna(), 'Charge', 'NoCharge')
df['Outcome_Monster'] = df['Outcome']+df['MonsterCharge']

df['v_reward_radial'] = df['v_reward_radial']*-1

df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()


In [ ]:
parts = []
for g_names, g, in df.groupby(['Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    outcome, session = g_names[0], g_names[1]
    #sampling + window
    fs  = float(1.0 / g['dt'].iloc[0])                 
    win = int(round(fs * 1.5)); win += 1 - (win % 2)  
    win = max(win, 3)

    if len(g) < win:
        continue

    # smooth position + derivative
    g['filtered_x']   = savgol_filter(g['X'].to_numpy(), win, 1, mode='interp')
    g['filtered_y']   = savgol_filter(g['Y'].to_numpy(), win, 1, mode='interp')
    dy                = savgol_filter(g['X'].to_numpy(), win, 1, deriv=1, delta=1/fs, mode='interp')
    g['d_filtered_x'] = lowpass(dy, fs, cutoff_hz=0.3)   
    g.loc[abs(g['d_filtered_x']) <2, 'd_filtered_x' ] = 0 

    #base labels (separate steps for clarity)
    g['approach'] = np.nan
    g['approach'] = g['approach'].astype('object')
    g.loc[g['filtered_x'] < 30,  'approach'] = 'Distal Approach' 
    g.loc[g['filtered_x'] >= 30, 'approach'] = 'Proximal Approach' 
    g.loc[g['filtered_x'] >= 43, 'approach'] = 'Past Spout'
    g.loc[g['d_filtered_x'] < 0, 'approach'] = 'Retreat' 

    #reward -> first retreat becomes 40
    rewardix = g['Time to Reward'].abs().idxmin()
    monsterix = (g['Time to Monster'].abs()).idxmin()
    consumptionix = np.nanmin([g.loc[(g.index > rewardix) & (g['approach'] == 'Retreat')].index.min(), g.loc[(g.index > rewardix) & (g['approach'] == 'Past Spout')].index.min()])
    g.loc[rewardix:consumptionix, 'approach'] = 'Licking'

    g.loc[(g['filtered_x'] <= 0), 'approach'] = np.nan

    lab  = g['approach'].to_numpy() 
    prev = np.concatenate([lab[:1], lab[:-1]]) 

    transitions = (prev == 'Retreat') & (lab != 'Retreat')
    g['approach_counter'] = np.cumsum(transitions.astype(int))

    parts.append(g)
    
df = pd.concat(parts, axis=0).reset_index(drop=True)

df = df[df['approach'].notna()]

parts = []
for gname, g in df.groupby(['approach_counter', 'Outcome_Monster', 'Session', 'Animal', 'Trial', 'Site']):
    rmin, rmax = np.min(g['Time to Reward']), np.max(g['Time to Reward'])
    mmin, mmax = np.min(g['Time to Monster']), np.max(g['Time to Monster'])
    if np.isnan(rmin):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax<0)):
        g['Reward'] = 'Unrewarded'
    elif ((rmin<0) & (rmax>0)):
        g['Reward'] = 'Rewarded'
    elif ((rmin>0) & (rmax>0)):
        g['Reward'] = 'PostRewarded'

    if np.isnan(mmin):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax<0)):
        g['Monster'] = 'NoCharge'
    elif ((mmin<0) & (mmax>0)):
        g['Monster'] = 'MonsterCharge'
    elif ((mmin>0) & (mmax>0)):
        g['Monster'] = 'MonsterCharging'

    parts.append(g)
df = pd.concat(parts, axis=0).reset_index(drop=True)

In [ ]:
df["Facing_Spout"] = "Side"
df.loc[(df['heading_relative_to_reward'] >= -20) & (df['heading_relative_to_reward'] <= 20), "Facing_Spout"] = "Front"
df.loc[(df['heading_relative_to_reward'] >= 90) | (df['heading_relative_to_reward'] <= -90), "Facing_Spout"] = "Behind"

df["Distance.to.Spout.fac"] = "Distal"
df.loc[(df['Distance to Spout'] <= 10), "Distance.to.Spout.fac"] = "Proximal"

df.rename(columns={'Distance to Spout': 'Distance.to.Spout'}, inplace=True)

df.loc[df["Site"] == "TSR", "heading_relative_to_reward"] *= -1
df.loc[
    (df["Site"] == "VS") & (~df["Animal"].str.contains("EI")),
    "heading_relative_to_reward"] *= -1

df['ipsi_contra'] = "Contra"
df.loc[(df['heading_relative_to_reward'] <= 0) , "ipsi_contra"] = "Ipsi"

In [ ]:
g = ["Region","Session", "Animal","Trial","Site"]
k = np.where(np.isclose(df.dt, 1/12, atol=1e-9), 12, 20)

num = [c for c in df.select_dtypes("number").columns if c not in g]
obj = [c for c in df.columns if c not in num+g]

df = (df.assign(_k=k, _i=df.groupby(g).cumcount(), bin=lambda d: d._i//d._k)
        .groupby(g+["bin"], as_index=False, observed=True)
        .agg({**{c:"mean" for c in num}, **{c:"first" for c in obj}})
        .drop(columns=["bin"])
        .assign(Z_lag1=lambda d: d.groupby(g)["Z"].shift(1))
     )

df = df[df['X']<=40].copy()

Figure S7A,S7E

In [ ]:
df_plot = df.copy()

REGION_ORDER  = ["VS", "TS"]
SESSION_ORDER = ["Reward", "Monster"]

ANIMAL_COL = "Animal"
x_col     = "heading_relative_to_reward"
y_col     = "Distance.to.Spout"
value_col = "Z"

NX = 15
NY = 40

X_RANGE = (-180,180)
Y_RANGE = (0.0, 40.0)

QLO, QHI = 1, 99

CENTER_AT_ZERO = True
VMIN_VMAX_Q = (2, 98)

CMAP = "plasma"

SQUARE_PANELS = True

FIGSIZE = (9.0, 8.2)

required = [ANIMAL_COL, "Region", "Session", x_col, y_col, value_col]
missing = [c for c in required if c not in df_plot.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df_plot["Region"]   = df_plot["Region"].astype(str)
df_plot["Session"]  = df_plot["Session"].astype(str)
df_plot[ANIMAL_COL] = df_plot[ANIMAL_COL].astype(str)

df_plot[x_col]     = pd.to_numeric(df_plot[x_col], errors="coerce")
df_plot[y_col]     = pd.to_numeric(df_plot[y_col], errors="coerce")
df_plot[value_col] = pd.to_numeric(df_plot[value_col], errors="coerce")

df_plot = df_plot[
    np.isfinite(df_plot[x_col]) &
    np.isfinite(df_plot[y_col]) &
    np.isfinite(df_plot[value_col])
].copy()

df_plot = df_plot[
    df_plot["Region"].isin(REGION_ORDER) &
    df_plot["Session"].isin(SESSION_ORDER)
].copy()

if X_RANGE is None:
    X_RANGE = tuple(np.nanpercentile(df_plot[x_col].to_numpy(), [QLO, QHI]))

ymin, ymax = Y_RANGE

xedges = np.linspace(X_RANGE[0], X_RANGE[1], NX + 1)
yedges = np.linspace(ymin, ymax, NY + 1)

df_plot["xbin"] = pd.cut(df_plot[x_col], bins=xedges, include_lowest=True, labels=False)
df_plot["ybin"] = pd.cut(df_plot[y_col], bins=yedges, include_lowest=True, labels=False)
df_plot = df_plot.dropna(subset=["xbin", "ybin"]).copy()
df_plot["xbin"] = df_plot["xbin"].astype(int)
df_plot["ybin"] = df_plot["ybin"].astype(int)

animal_bin = (
    df_plot
    .groupby(["Session", "Region", ANIMAL_COL, "ybin", "xbin"], observed=True)[value_col]
    .mean()
    .reset_index()
)

bin_mean = (
    animal_bin
    .groupby(["Session", "Region", "ybin", "xbin"], observed=True)[value_col]
    .mean()
    .reset_index()
)

mats = {}
all_vals = []

for sess in SESSION_ORDER:
    for reg in REGION_ORDER:
        sub = bin_mean[(bin_mean["Session"] == sess) & (bin_mean["Region"] == reg)]
        mat = np.full((NY, NX), np.nan, dtype=float)
        if len(sub):
            mat[sub["ybin"].to_numpy(), sub["xbin"].to_numpy()] = sub[value_col].to_numpy()
        mats[(sess, reg)] = mat
        all_vals.append(mat.ravel())

all_vals = np.concatenate(all_vals)
all_vals = all_vals[np.isfinite(all_vals)]
if all_vals.size == 0:
    raise ValueError("No finite binned values to plot (check filters / ranges).")

vmin, vmax = np.nanpercentile(all_vals, VMIN_VMAX_Q)
norm = mpl.colors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax) if CENTER_AT_ZERO else mpl.colors.Normalize(vmin=vmin, vmax=vmax)

extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]

fig, axes = plt.subplots(
    nrows=len(SESSION_ORDER),
    ncols=len(REGION_ORDER),
    figsize=FIGSIZE,
    constrained_layout=True,
    sharex=True,
    sharey=True
)

last_im = None
for i, sess in enumerate(SESSION_ORDER):
    for j, reg in enumerate(REGION_ORDER):
        ax = axes[i, j]

        if SQUARE_PANELS:
            ax.set_box_aspect(1)

        mat = mats[(sess, reg)]

        last_im = ax.imshow(
            mat,
            origin="lower",
            extent=extent,
            aspect="auto",
            cmap=CMAP,
            norm=norm,
            interpolation="nearest",
        )

        ax.set_title(f"Session = {sess} | Region = {reg}", fontsize=10)

        if i == len(SESSION_ORDER) - 1:
            ax.set_xlabel(x_col)
        if j == 0:
            ax.set_ylabel(y_col)

        ax.invert_yaxis()

        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(5))

cbar = fig.colorbar(last_im, ax=axes, fraction=0.04, pad=0.02)
cbar.set_label(f"Mean {value_col} (animal-averaged)", fontsize=10)

plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\Heatmap_AngleDist_AvgRawData.svg')

plt.show()